# 🧩 Mini-Lab: Build the CLI: Arguments, Exit Codes, stdout vs stderr

**Module: Delivery: CLI, API, and Debugging Lab 2** | **Type: Mini-Lab (Brick)**

---

## Learning Objectives

By the end of this mini-lab, you will be able to:

1. **Explain** why a CLI is the delivery interface for this project
2. **Describe** a CLI contract: arguments, `--help`, exit codes, and stdout versus stderr
3. **Recognize** how a flag, an environment variable, and a built-in default layer together
4. **Identify** what a subprocess integration test of the shipped entry point catches

## Target Concepts

| Concept | Gloss |
|---------|-------|
| CLI contract | A CLI contract defines how a command is used and how it behaves: its arguments and flags, what `--help` promises, which exit codes mean success or failure, and what goes to stdout versus stderr. |
| Layered configuration | Settings resolve in a fixed order: built-in defaults, then environment variables, then explicit flags |
| Integration test | Exercise the shipped entry point, not just internal functions, so you catch real wiring issues across components, config, and dependencies. |
| Workflow form choice | Pick the delivery form (notebook, library, CLI, API) that fits how the work will be used |

## Why a CLI?

Our core is a small **triage** function. It labels a support message as `high` or `normal` priority. Before we write any code, we choose the **workflow form** used to deliver it:

| Form | Good for | Why not here |
|------|----------|--------------|
| Notebook | Exploring | A notebook is not a command that other people or scripts can run |
| Library | Other Python code | A shell user or a cron job cannot call it directly |
| HTTP API | Remote clients | It needs a server, ports, and uptime (optional later) |
| **CLI** | **Humans, scripts, CI** | **This is the required interface: one command with no server, and a subprocess can test it** |

We pick the **CLI**.

## Step 1: Write the Entry Point

The file below *is* the contract:

- **Arguments and help:** a required `text`, an optional `--format {text,json}`, and a `--help` that lists the exit codes
- **Exit codes:** `0` = success, `1` = empty input, `2` = usage error (argparse sets this one)
- **Streams:** results go to **stdout**, and errors go to **stderr**

Config order: `--format` flag > `TRIAGE_FORMAT` env var > built-in default `text`.

> 🧩 **CLI contract.** The contract from the video, now written as code.

In [1]:
%%writefile triage_cli.py
"""Triage a support message from the command line."""
import argparse
import json
import os
import sys

URGENT = ("down", "outage", "refund")


def triage(text):
    urgent = any(word in text.lower() for word in URGENT)
    return {"priority": "high" if urgent else "normal"}


def main(argv=None):
    parser = argparse.ArgumentParser(
        prog="triage",
        description="Classify a support message as high or normal priority.",
        epilog="Exit codes: 0 = success, 1 = empty input, 2 = usage error.",
    )
    parser.add_argument("text", help="the support message to triage")
    parser.add_argument(
        "--format",
        choices=["text", "json"],
        default=os.environ.get("TRIAGE_FORMAT", "text"),
        help="output format (default: $TRIAGE_FORMAT, else text)",
    )
    args = parser.parse_args(argv)

    if not args.text.strip():
        print("error: message is empty", file=sys.stderr)
        return 1

    result = triage(args.text)
    if args.format == "json":
        print(json.dumps(result))
    else:
        print(f"priority: {result['priority']}")
    return 0


if __name__ == "__main__":
    sys.exit(main())

Overwriting triage_cli.py


## Step 2: Read What `--help` Promises

Run `--help` through a subprocess to see the contract users get. `sys.executable` makes the same call work on Mac and Windows.

In [2]:
import os
import subprocess
import sys

os.environ.pop("TRIAGE_FORMAT", None)  # start from a clean environment

def run_cli(*args, **env):
    return subprocess.run(
        [sys.executable, "triage_cli.py", *args],
        capture_output=True, text=True, env={**os.environ, **env},
    )

help_run = run_cli("--help")
help_lines = help_run.stdout.splitlines()

# The usage line comes first, and the epilog lists the exit codes
print(help_lines[0])
print(next((line for line in help_lines if "Exit codes" in line), "(no exit-codes line found)"))
print("exit code:", help_run.returncode)

usage: triage [-h] [--format {text,json}] text
Exit codes: 0 = success, 1 = empty input, 2 = usage error.
exit code: 0


## Step 3: Exit Codes and stdout vs stderr

Run one success and two kinds of failure. Watch which stream each message lands in and which exit code comes back.

In [3]:
def show(r):
    # Print a short snippet of each stream, or (empty)
    out = r.stdout.strip()[:60] or "(empty)"
    err = r.stderr.strip()[:60] or "(empty)"
    print(f"exit={r.returncode} | stdout: {out} | stderr: {err}")

show(run_cli("The site is down!"))          # success
show(run_cli("   "))                        # our own failure: empty input
show(run_cli("hi", "--format", "xml"))      # usage error caught by argparse

exit=0 | stdout: priority: high | stderr: (empty)


exit=1 | stdout: (empty) | stderr: error: message is empty
exit=2 | stdout: (empty) | stderr: usage: triage [-h] [--format {text,json}] text
triage: error


## Step 4: Layered Configuration

The same command resolves `--format` from three layers. A layer higher in the list overrides the layers below it:

1. `--format` flag
2. `TRIAGE_FORMAT` environment variable
3. Built-in default `text`

In [4]:
print("1) built-in default only")
show(run_cli("Refund please"))

print("2) env var overrides default")
show(run_cli("Refund please", TRIAGE_FORMAT="json"))

print("A --format flag would override the env var in turn: flag > env > default")

1) built-in default only


exit=0 | stdout: priority: high | stderr: (empty)
2) env var overrides default


exit=0 | stdout: {"priority": "high"} | stderr: (empty)
A --format flag would override the env var in turn: flag > env > default


## Step 5: Integration-Test the Shipped Entry Point

Two tests lock the contract in place. Neither one imports `triage()` directly. Each test launches the CLI exactly as a user would and checks the exit code, stdout, and stderr.

> 🧩 **Integration test.** Watch for the tests running `triage_cli.py` as a separate process. Argument parsing, the env-var layer, exit codes, and both streams are all exercised together.

In [5]:
%%writefile test_triage_cli.py
import json
import os
import subprocess
import sys
from pathlib import Path

CLI = Path(__file__).parent / "triage_cli.py"


def run(*args, **env):
    return subprocess.run(
        [sys.executable, str(CLI), *args],
        capture_output=True, text=True, env={**os.environ, **env},
    )


def test_urgent_message_prints_json_to_stdout():
    r = run("Our API is down", "--format", "json")
    assert r.returncode == 0
    assert json.loads(r.stdout) == {"priority": "high"}
    assert r.stderr == ""


def test_empty_message_exits_1_with_error_on_stderr():
    r = run("   ")
    assert r.returncode == 1
    assert r.stdout == ""
    assert "empty" in r.stderr

Overwriting test_triage_cli.py


In [6]:
pytest_run = subprocess.run(
    [sys.executable, "-m", "pytest", "-q", "-p", "no:cacheprovider", "test_triage_cli.py"],
    capture_output=True, text=True,
)
# The last line of pytest output is the summary, e.g. "2 passed in 0.31s"
print(pytest_run.stdout.strip().splitlines()[-1])

2 passed in 0.36s


## 🎯 Summary

### Key Takeaways

1. **CLI as the delivery form** — one command serves humans, scripts, and CI without needing a server
2. **CLI contract** — arguments and `--help` define usage, and exit codes `0`/`1`/`2` signal the outcome
3. **stdout vs stderr** — results go to stdout and errors go to stderr, so pipes and scripts stay clean
4. **Layered configuration** — a flag overrides an env var, and an env var overrides the built-in default
5. **Integration tests** — subprocess tests of the shipped entry point catch wiring bugs that unit tests miss